# Pydantic model to structured output

TODO: skapa en agent som ska simulera en IT-anställd

In [3]:
from pydantic_ai import Agent
from dotenv import load_dotenv

load_dotenv()
# översätt till engelska
employee_agent = Agent(
    "openrouter:liquid/lfm-2.5-1.2b-thinking:free",
    system_prompt="""You are an IT employee who helps with various IT problems. You are very quick and efficient..
    
    Areas to include in the output:
    - name
    - age
    - gender
    - job title
    - salary SEK per month """,
)

prompt = "Simulate 2 employees"

result = await employee_agent.run(prompt)

print(result.output)


Here are two simulated employees:  

1. **Name**: James  
   **Age**: 29  
   **Gender**: Male  
   **Job Title**: Software Engineer  
   **Salary**: 800000 SEK  

2. **Name**: Maria  
   **Age**: 31  
   **Gender**: Female  
   **Job Title**: Designer  
   **Salary**: 670000 SEK  

Both are in their late 20s/early 30s, typical IT roles. Let me know if you need adjustments!


In [4]:
with open("simulated_employees.md", "w") as file:
    file.write(result.output)

## Get more structured output

issue with above:
- output structure vary
- hard to work with the data e.g. compute mean of salaries

want:
- repeatable structure

In [5]:
from pydantic import BaseModel, Field
from typing import Literal


class EmployeeModel(BaseModel):
    name: str = Field(description="Mostly swedish names, but could be foreign as well")
    age: int = Field(description="Age should be between 18 and 67")
    gender: Literal["Male", "Female"]
    experience_level: Literal["Entry", "Mid level", "Senior", "Expert"]
    job_title: str
    salary: int = Field(
        ge=30_000,
        le=50_000,
        description="Salary should be between 30k and 50k, the higher the experience level, the higher salary",
    )


employee_simulator_agent = Agent(
    "openrouter:arcee-ai/trinity-mini:free",
    system_prompt="You are an HR expert within IT field in Sweden within data science, data engineering, machine learning, AI engineering. You will simulate IT employees",
    retries=1,
)

result = await employee_simulator_agent.run(
    "Give me 3 employees", output_type=EmployeeModel
)
result

AgentRunResult(output=EmployeeModel(name='Lars', age=30, gender='Male', experience_level='Entry', job_title='Data Scientist', salary=35000))

In [6]:
result.output

EmployeeModel(name='Lars', age=30, gender='Male', experience_level='Entry', job_title='Data Scientist', salary=35000)

In [7]:
result.output.salary + 5000

40000

In [8]:
result = await employee_simulator_agent.run(
    "Give me 4 employees, one named Lucas Lindh", output_type=list[EmployeeModel]
)
result

AgentRunResult(output=[EmployeeModel(name='Lucas Lindh', age=30, gender='Male', experience_level='Senior', job_title='Senior Data Scientist', salary=50000), EmployeeModel(name='Anna Svensson', age=28, gender='Female', experience_level='Mid level', job_title='Mid level Data Engineer', salary=40000), EmployeeModel(name='Erik Johansson', age=35, gender='Male', experience_level='Entry', job_title='Entry level AI Engineer', salary=30000), EmployeeModel(name='Maria Andersson', age=32, gender='Female', experience_level='Senior', job_title='Senior Machine Learning Engineer', salary=45000)])

In [9]:
result.output[0].model_dump()

{'name': 'Lucas Lindh',
 'age': 30,
 'gender': 'Male',
 'experience_level': 'Senior',
 'job_title': 'Senior Data Scientist',
 'salary': 50000}

TODO: 
- result.output make into list of dictionaries
- create pandas dataframe based on this list
- export to csv file of our simulated employees

In [10]:
employees = result.output

employees_list = [employee.model_dump() for employee in employees]
employees_list

import pandas as pd

df = pd.DataFrame(employees_list)

In [11]:
df

,name,age,gender,experience_level,job_title,salary
0,Lucas Lindh,30,Male,Senior,Senior Data Scientist,50000
1,Anna Svensson,28,Female,Mid level,Mid level Data Engineer,40000
2,Erik Johansson,35,Male,Entry,Entry level AI Engineer,30000
3,Maria Andersson,32,Female,Senior,Senior Machine Learning Engineer,45000


In [12]:
df
df.dtypes
df["salary"].mean()

np.float64(41250.0)

In [13]:
df.to_csv("simulated_employees.csv", index=False)

# Task

Övning:

Gå från ostrukturerad text till strukturerad outputs. 

1. MLOps Engineer
plocka ut följande fält:
- education_name
- program_description (summary)
- career_options (list of jobs)
- course_list
- length

2. Kontakta oss
plocka ut:
- department
- opening_hours
- phone_time
- phone

exportera 2 st json-filer

tips: 
- skapa pydantic model
- system prompt där man beskriver att den ska plocka ut information från text
- i agent.run  skicka med själva texten

In [ ]:
from pathlib import Path
import json

# mlops_text = Path("mlops_raw.txt").read_text()
# contact_us_text = Path("nackademin_contacts.txt").read_text()

with open("data/mlops_raw.txt", "r") as file:
    mlops_text = file.read()

with open("data/nackademin_contacts.txt", "r") as file:
    contact_us_text = file.read()

mlops_prompt = f"You will be given a text about the MLOps Engineer education program. Your job is to extract the information and return it in a structured way."
contact_us_prompt = f"You will be given a text about the contact information for Nackademin. Your job is to extract the information and return it in a structured way."

class MLOpsEngineer(BaseModel):
    education_name: str = Field(description="Name of the education program")
    program_description: str = Field(description="Summary of the education program")
    career_options: list[str] = Field(description="List of jobs that can be pursued after the education")
    course_list: list[str] = Field(description="List of courses in the education program")
    length: float = Field(description="Length of the education program in weeks")

class ContactUs(BaseModel):
    department: str = Field(description="Department of the contact")
    opening_hours: str = Field(description="Opening hours of the contact")
    phone_time: str = Field(description="Phone time of the contact")
    phone: str = Field(description="Phone number of the contact")

mlops_engineer_agent = Agent(
    "openrouter:arcee-ai/trinity-mini:free",
    system_prompt=contact_us_prompt,
    retries=1,
)


result = await mlops_engineer_agent.run(contact_us_text, output_type=ContactUs)

result.output.model_dump()

{'department': 'Generella frågor',
 'opening_hours': 'kl. 08.00 – 12.00',
 'phone_time': 'kl. 08.00 – 12.00',
 'phone': '08-466 60 00'}

In [34]:
result.output.model_dump_json()

'{"department":"Generella frågor","opening_hours":"kl. 08.00 – 12.00","phone_time":"kl. 08.00-12.00","phone":"08-466 60 00"}'

In [35]:
with open("contact_us.json", "w") as file:
    file.write(result.output.model_dump_json())


In [36]:
mlops_text

'Vill du jobba i skärningspunkten mellan AI, data och teknik? Som MLOps Engineer lär du dig att bygga, driftsätta och optimera maskininlärningsmodeller i verkligheten. Utbildningen passar dig som har en teknisk bakgrund och vill ta nästa steg mot en framtidsroll där du gör avancerade AI-lösningar möjliga.\nOm programmet\n \n\nAI och maskininlärning används i allt fler branscher och behovet av specialister som kan ta modellerna från idé till drift växer snabbt. Som student får du spetskompetens inom maskininlärning och MLOps. Du arbetar med verktyg som TensorFlow och PyTorch och lär dig allt från CI/CD-pipelines och versionshantering till säkerhet och edge computing.\n\n \n\nUtbildningen bedrivs främst på svenska, men mycket kursmaterial är på engelska. Det gör att du förbereds för den internationella arbetsmiljö som präglar IT och AI-utveckling. Efter examen kan du självständigt driftsätta, optimera och förvalta ML-modeller, och bidra till nästa generations AI-lösningar.\n\n \n\nFördel